# 📖 Notebook 4: Scaling Elasticsearch Clusters

So far we've used Elasticsearch as a single-node search engine. In production,
Elasticsearch is a **distributed system** — data is split across multiple nodes
for performance and fault tolerance.

This notebook explores how Elasticsearch scales, from the high-level cluster
architecture all the way down to Lucene segments.

## Learning Objectives

By the end of this notebook, you'll understand:
- How **shards** split data across nodes (horizontal scaling)
- How **replicas** provide fault tolerance and read throughput
- The different **node types** and their roles
- How **Lucene segments** work under the hood (immutable, merged, etc.)
- How the **inverted index** makes full-text search fast
- Performance considerations for real-world clusters

## 🛠️ Setup

```bash
cd deep-dives/elasticsearch
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
import json
import time

es = Elasticsearch("http://localhost:9200")
info = es.info()
print(f"✅ Connected to Elasticsearch {info['version']['number']}")

## 🏗️ Cluster Architecture Overview

An Elasticsearch cluster is made up of **nodes** — individual server instances.
Each node has one or more **roles**:

```
┌──────────────────────────────────────────────────────────────┐
│                    ELASTICSEARCH CLUSTER                     │
│                                                              │
│  ┌─────────────┐  ┌─────────────┐  ┌─────────────┐         │
│  │  Master Node │  │  Data Node  │  │  Data Node  │         │
│  │  (admin)     │  │  (storage)  │  │  (storage)  │         │
│  └─────────────┘  └─────────────┘  └─────────────┘         │
│                                                              │
│  ┌──────────────────┐  ┌─────────────┐                      │
│  │ Coordinating Node │  │ Ingest Node │                      │
│  │ (query routing)   │  │ (transform) │                      │
│  └──────────────────┘  └─────────────┘                      │
└──────────────────────────────────────────────────────────────┘
```

| Node Type | Role | Analogy |
|-----------|------|--------|
| **Master** | Cluster admin — creates/deletes indices, tracks which nodes have which data | The manager |
| **Data** | Stores data and executes searches on it | The warehouse workers |
| **Coordinating** | Routes client requests to the right data nodes, merges results | The receptionist |
| **Ingest** | Pre-processes documents before indexing (transforms, enrichment) | The mail room |

In production, different node types run on different hardware:
- **Data nodes** → lots of disk and memory (they store everything)
- **Coordinating nodes** → good network and CPU (they merge results)
- **Master nodes** → modest resources (they just coordinate)

In [ ]:
# Let's inspect our cluster (it's just one node in Docker, but the APIs are the same)
print("🏗️ Cluster Health")
print("=" * 50)

health = es.cluster.health()
print(f"  Cluster name:         {health['cluster_name']}")
print(f"  Status:               {health['status']}")
print(f"  Number of nodes:      {health['number_of_nodes']}")
print(f"  Active shards:        {health['active_shards']}")
print(f"  Unassigned shards:    {health['unassigned_shards']}")

status_emoji = {"green": "🟢", "yellow": "🟡", "red": "🔴"}
print(f"\n{status_emoji.get(health['status'], '⚪')} Cluster status explained:")
print("  🟢 Green:  All shards and replicas are assigned")
print("  🟡 Yellow: All primary shards are assigned, some replicas are not")
print("  🔴 Red:    Some primary shards are not assigned (data loss risk!)")
print("\n💡 A single-node cluster is typically 'yellow' or 'green' because")
print("   replicas can't be placed on the same node as their primary shard.")

In [ ]:
# Node information
print("🖥️ Node Details")
print("=" * 60)

nodes = es.nodes.info()
for node_id, node_info in nodes["nodes"].items():
    print(f"  Node ID:     {node_id[:12]}...")
    print(f"  Name:        {node_info['name']}")
    print(f"  Roles:       {', '.join(node_info['roles'])}")
    print(f"  OS:          {node_info['os']['name']}")
    print(f"  JVM Heap:    {node_info['jvm']['mem']['heap_max_in_bytes'] / 1024 / 1024:.0f} MB")
    print()

## 📦 Shards: Splitting Data Across Nodes

A **shard** is a piece of an index. When you create an index, you choose
how many shards to split it into.

```
         Index: "books" (3 shards)
    ┌──────────┬──────────┬──────────┐
    │ Shard 0  │ Shard 1  │ Shard 2  │
    │ (books   │ (books   │ (books   │
    │  A–G)    │  H–O)    │  P–Z)    │
    └──────────┴──────────┴──────────┘
         ↓           ↓           ↓
      Node 1      Node 2      Node 3
```

**Why shard?**
- **Horizontal scaling**: Each shard can live on a different node
- **Parallel search**: Queries run on all shards simultaneously
- **Capacity**: A single node has limits; sharding lets you go beyond them

**How many shards?**
- Too few → can't distribute load, one node becomes a bottleneck
- Too many → overhead per shard (memory, file handles), slower coordinating
- Rule of thumb: aim for shards between 10GB and 50GB each

In [ ]:
# Create indices with different shard counts to observe the effect
for idx_name, num_shards in [("demo_1shard", 1), ("demo_3shards", 3), ("demo_5shards", 5)]:
    if es.indices.exists(index=idx_name):
        es.indices.delete(index=idx_name)
    es.indices.create(
        index=idx_name,
        settings={"number_of_shards": num_shards, "number_of_replicas": 0}
    )

print("📦 Shard allocation across indices:")
print("=" * 60)

# Show shard information
for idx_name in ["demo_1shard", "demo_3shards", "demo_5shards"]:
    settings = es.indices.get_settings(index=idx_name)
    shards = settings[idx_name]["settings"]["index"]["number_of_shards"]
    replicas = settings[idx_name]["settings"]["index"]["number_of_replicas"]
    total = int(shards) * (1 + int(replicas))
    print(f"  {idx_name:<20} {shards} primary shard(s) × {1 + int(replicas)} = {total} total shard(s)")

print("\n💡 Each shard is a full Lucene index. More shards = more parallel search")
print("   capacity, but also more memory overhead.")

## 🔄 Replicas: Fault Tolerance + Read Throughput

A **replica** is an exact copy of a primary shard, stored on a different node.

```
         Index: "books" (2 shards, 1 replica)
    ┌──────────────────┬──────────────────┐
    │ Node 1           │ Node 2           │
    │                  │                  │
    │  [P0] primary    │  [P1] primary    │
    │  [R1] replica    │  [R0] replica    │
    └──────────────────┴──────────────────┘
```

**Why replicas?**
1. **High Availability**: If a node dies, the replica takes over
2. **Read Throughput**: Search requests can be served by any copy (primary or replica)

If a shard handles X queries/sec, having Y replicas gives you X × (Y+1) queries/sec.

⚠️ **Important**: Replicas can't be on the same node as their primary.
On a single-node cluster, replicas stay "unassigned" (yellow status).

In [ ]:
# Create an index with replicas and observe the cluster health
TEST_INDEX = "replica_demo"

if es.indices.exists(index=TEST_INDEX):
    es.indices.delete(index=TEST_INDEX)

# 1 shard, 0 replicas → should be green (even on single node)
es.indices.create(index=TEST_INDEX, settings={"number_of_shards": 1, "number_of_replicas": 0})
health = es.cluster.health(index=TEST_INDEX)
print(f"🟢 0 replicas: status = {health['status']}")
print(f"   Active shards: {health['active_shards']}, Unassigned: {health['unassigned_shards']}")

# Now increase to 1 replica
es.indices.put_settings(index=TEST_INDEX, settings={"number_of_replicas": 1})
health = es.cluster.health(index=TEST_INDEX)
print(f"\n🟡 1 replica: status = {health['status']}")
print(f"   Active shards: {health['active_shards']}, Unassigned: {health['unassigned_shards']}")
print("   ↑ Replica can't be assigned on a single-node cluster!")

# Reset to 0 replicas
es.indices.put_settings(index=TEST_INDEX, settings={"number_of_replicas": 0})
print("\n💡 In production with multiple nodes, replicas would be green.")

## 🔍 How Search Works Across Shards

When you search an index with multiple shards, Elasticsearch uses a
**scatter-gather** pattern:

```
  Client → Coordinating Node
              │
              ├──→ Shard 0: search locally, return top N results
              ├──→ Shard 1: search locally, return top N results
              └──→ Shard 2: search locally, return top N results
              │
              ← Merge all results, re-rank, return top N to client
```

This is called the **query-then-fetch** model:
1. **Query phase**: Each shard finds matching document IDs and scores
2. **Fetch phase**: The coordinating node asks only the relevant shards for the full documents

Let's see this in action by indexing data into a multi-shard index.

In [ ]:
# Create a 3-shard index and index some data
MULTI_INDEX = "multi_shard_demo"

if es.indices.exists(index=MULTI_INDEX):
    es.indices.delete(index=MULTI_INDEX)

es.indices.create(
    index=MULTI_INDEX,
    settings={"number_of_shards": 3, "number_of_replicas": 0},
    mappings={"properties": {
        "title": {"type": "text"},
        "category": {"type": "keyword"},
        "price": {"type": "float"}
    }}
)

# Bulk index 100 documents
import random
random.seed(42)
categories = ["Electronics", "Books", "Clothing", "Home", "Sports"]
adjectives = ["Amazing", "Premium", "Budget", "Deluxe", "Classic", "Modern", "Essential"]
nouns = ["Widget", "Gadget", "Tool", "Device", "Kit", "Set", "Pack"]

actions = []
for i in range(100):
    actions.append({
        "_index": MULTI_INDEX,
        "_id": i+1,
        "_source": {
            "title": f"{random.choice(adjectives)} {random.choice(nouns)} {i+1}",
            "category": random.choice(categories),
            "price": round(random.uniform(5, 200), 2)
        }
    })

bulk(es, actions)
es.indices.refresh(index=MULTI_INDEX)

# Show how documents are distributed across shards
print("📦 Document Distribution Across 3 Shards")
print("=" * 50)

# Use _search_shards to see shard allocation
shard_stats = es.search(
    index=MULTI_INDEX,
    size=0,
    aggs={
        "by_shard": {
            "terms": {"field": "_shard", "size": 10}
        }
    }
)

total_docs = es.count(index=MULTI_INDEX)["count"]
print(f"  Total documents: {total_docs}")
print(f"  Distributed across 3 shards")
print(f"\n💡 Elasticsearch uses a hash of the document ID to decide which shard")
print(f"   gets each document: shard = hash(_id) % number_of_shards")
print(f"   This ensures roughly even distribution.")

In [ ]:
# Search the multi-shard index — notice the _shards info in the response
print("🔍 Search Across Multiple Shards")
print("=" * 50)

results = es.search(
    index=MULTI_INDEX,
    query={"match": {"title": "premium"}},
    size=5
)

shards = results["_shards"]
print(f"  Shards queried:  {shards['total']}")
print(f"  Successful:      {shards['successful']}")
print(f"  Failed:          {shards['failed']}")
print(f"  Results found:   {results['hits']['total']['value']}")
print(f"  Time taken:      {results['took']}ms")
print(f"\n  Top results:")
for hit in results["hits"]["hits"][:5]:
    print(f"    {hit['_source']['title']} (${hit['_source']['price']:.2f})")

print(f"\n💡 All 3 shards were queried in parallel. The coordinating node")
print(f"   merged the results and returned the top matches.")

## 📚 Lucene Internals: Segments

Each shard is a **Lucene index**, and each Lucene index is made up of
**segments** — immutable containers of indexed data.

```
Shard 0 (Lucene Index)
  ├── Segment A (100 docs, created at t=0)
  ├── Segment B (50 docs, created at t=1)
  └── Segment C (75 docs, created at t=2)
```

### Key Properties of Segments:

1. **Immutable**: Once written, a segment never changes
2. **Batched writes**: New documents are buffered, then flushed as a new segment
3. **Deletes are soft**: Deleted docs are marked in a separate file, not removed
4. **Updates = delete + insert**: Updating a doc marks the old version as deleted and creates a new one
5. **Merged periodically**: Small segments are merged into larger ones (cleanup happens here)

### Why Immutability?

| Benefit | Why |
|---------|----|
| Fast writes | Just append a new segment — no modifying existing data |
| Safe caching | Segments never change, so caches are always valid |
| No locking | Readers don't compete with writers |
| Easy recovery | Known, consistent state after crashes |
| Better compression | Static data compresses more efficiently |

In [ ]:
# Let's observe segments forming in real time!
SEGMENT_INDEX = "segment_demo"

if es.indices.exists(index=SEGMENT_INDEX):
    es.indices.delete(index=SEGMENT_INDEX)

es.indices.create(
    index=SEGMENT_INDEX,
    settings={
        "number_of_shards": 1,
        "number_of_replicas": 0,
        "refresh_interval": "-1"  # disable auto-refresh to control segment creation
    }
)

def show_segments(index):
    """Display segment information for an index."""
    segments = es.indices.segments(index=index)
    for shard_id, shard_data in segments["indices"][index]["shards"].items():
        for shard in shard_data:
            segs = shard["segments"]
            print(f"  Shard {shard_id}: {len(segs)} segment(s)")
            for seg_name, seg_info in segs.items():
                print(f"    └── {seg_name}: {seg_info['num_docs']} docs, "
                      f"{seg_info['deleted_docs']} deleted, "
                      f"size: {seg_info['size_in_bytes']} bytes")

# Step 1: Index some documents and force a refresh (creates segment 1)
for i in range(10):
    es.index(index=SEGMENT_INDEX, document={"title": f"Book {i}", "price": i * 10})
es.indices.refresh(index=SEGMENT_INDEX)

print("📚 After indexing 10 docs + refresh:")
show_segments(SEGMENT_INDEX)

# Step 2: Index more documents and refresh again (creates segment 2)
for i in range(10, 20):
    es.index(index=SEGMENT_INDEX, document={"title": f"Book {i}", "price": i * 10})
es.indices.refresh(index=SEGMENT_INDEX)

print("\n📚 After indexing 10 more docs + refresh:")
show_segments(SEGMENT_INDEX)

# Step 3: Delete a document (soft delete — stays in segment until merge)
es.index(index=SEGMENT_INDEX, id="to_delete", document={"title": "Delete Me", "price": 0})
es.indices.refresh(index=SEGMENT_INDEX)
es.delete(index=SEGMENT_INDEX, id="to_delete")
es.indices.refresh(index=SEGMENT_INDEX)

print("\n📚 After adding + deleting a document:")
show_segments(SEGMENT_INDEX)
print("\n💡 Notice 'deleted_docs' — the document is marked deleted but still")
print("   takes space until segments are merged.")

In [ ]:
# Force a segment merge — this cleans up deleted documents
print("🔄 Forcing a segment merge...")
es.indices.forcemerge(index=SEGMENT_INDEX, max_num_segments=1)
es.indices.refresh(index=SEGMENT_INDEX)

print("\n📚 After force merge (1 segment):")
show_segments(SEGMENT_INDEX)
print("\n💡 All documents are now in a single segment. Deleted docs are gone!")
print("   In production, Elasticsearch merges automatically in the background.")

## 🔤 Inside a Segment: The Inverted Index

The **inverted index** is the data structure that makes full-text search fast.
Instead of scanning every document for a word, we look up the word in a
dictionary and immediately get the list of documents that contain it.

### Normal Index (Forward)

```
Document 1: "The Great Gatsby"  → [the, great, gatsby]
Document 2: "Great Expectations" → [great, expectations]
Document 3: "The Hobbit"         → [the, hobbit]
```

To find "great", you'd scan ALL documents → O(n)

### Inverted Index

```
"the"          → [Doc 1, Doc 3]
"great"        → [Doc 1, Doc 2]    ← instant lookup!
"gatsby"       → [Doc 1]
"expectations" → [Doc 2]
"hobbit"       → [Doc 3]
```

To find "great", look it up in the dictionary → O(1)

This is how Elasticsearch searches billions of documents in milliseconds.
Each text field in each segment has its own inverted index.

In [ ]:
# Demonstrate the inverted index concept with term vectors
# Term vectors show exactly what tokens are stored for a document
INVERTED_INDEX = "inverted_index_demo"

if es.indices.exists(index=INVERTED_INDEX):
    es.indices.delete(index=INVERTED_INDEX)

es.indices.create(
    index=INVERTED_INDEX,
    settings={"number_of_shards": 1, "number_of_replicas": 0},
    mappings={"properties": {
        "title": {"type": "text", "term_vector": "yes"},
        "description": {"type": "text", "term_vector": "with_positions_offsets"}
    }}
)

docs = [
    {"title": "The Great Gatsby", "description": "A novel about the American Dream"},
    {"title": "Great Expectations", "description": "A coming of age story about ambition"},
    {"title": "The Hobbit", "description": "A fantasy adventure about a hobbit's journey"},
]

for i, doc in enumerate(docs):
    es.index(index=INVERTED_INDEX, id=i+1, document=doc)
es.indices.refresh(index=INVERTED_INDEX)

# Show term vectors for document 1
print("🔤 Term Vectors for 'The Great Gatsby'")
print("=" * 50)

tv = es.termvectors(index=INVERTED_INDEX, id=1, fields=["title"])
if "title" in tv.get("term_vectors", {}):
    terms = tv["term_vectors"]["title"]["terms"]
    print(f"  Tokens stored in the inverted index for 'title':")
    for term, info in sorted(terms.items()):
        print(f"    '{term}' → appears {info['term_freq']} time(s)")

print("\n💡 The title 'The Great Gatsby' was tokenized and lowercased.")
print("   Each token points back to this document in the inverted index.")

## 📊 Doc Values: Fast Sorting and Aggregations

The inverted index is great for finding documents, but what about **sorting**
and **aggregations**? If we need to sort 10,000 results by price, we need
fast access to the price of every matched document.

This is where **doc values** come in — a columnar data structure stored
alongside each segment:

```
Inverted Index (for searching):     Doc Values (for sorting/aggregating):
  'great' → [Doc 1, Doc 2]          Doc 1: price = 9.99
  'gatsby' → [Doc 1]                Doc 2: price = 7.99
  ...                               Doc 3: price = 11.99
```

Doc values store each field's data contiguously (like a column in a
spreadsheet), making sorting and aggregations very cache-friendly.

## ⚡ Performance Considerations

Let's explore some key performance factors for Elasticsearch clusters.

In [ ]:
# Performance comparison: refresh interval impact
PERF_INDEX = "perf_demo"

def benchmark_indexing(index_name, refresh_interval, num_docs=500):
    """Benchmark indexing speed with different refresh intervals."""
    if es.indices.exists(index=index_name):
        es.indices.delete(index=index_name)

    es.indices.create(
        index=index_name,
        settings={
            "number_of_shards": 1,
            "number_of_replicas": 0,
            "refresh_interval": refresh_interval
        }
    )

    actions = [
        {"_index": index_name, "_source": {"title": f"Doc {i}", "value": i}}
        for i in range(num_docs)
    ]

    start = time.time()
    bulk(es, actions)
    elapsed = time.time() - start

    # Force a final refresh to make all docs searchable
    es.indices.refresh(index=index_name)
    count = es.count(index=index_name)["count"]

    return elapsed, count

print("⚡ Indexing Performance: Refresh Interval Impact")
print("=" * 55)
print(f"  Indexing 500 documents with different refresh intervals:\n")

for interval in ["1s", "5s", "30s", "-1"]:
    label = interval if interval != "-1" else "disabled"
    elapsed, count = benchmark_indexing(f"perf_{label}", interval)
    docs_per_sec = count / elapsed
    print(f"  refresh_interval={label:<10}  {elapsed:.3f}s  ({docs_per_sec:.0f} docs/sec)")

print("\n💡 Less frequent refreshes = faster indexing (fewer segments created).")
print("   Trade-off: documents take longer to become searchable.")
print("   For bulk imports, disable refresh entirely, then re-enable after.")

In [ ]:
# Index stats: see storage, segments, and memory usage
print("📊 Index Statistics")
print("=" * 60)

for idx_name in ["perf_1s", "perf_disabled"]:
    if not es.indices.exists(index=idx_name):
        continue
    stats = es.indices.stats(index=idx_name)
    idx_stats = stats["indices"][idx_name]["total"]

    print(f"\n  Index: {idx_name}")
    print(f"    Documents:    {idx_stats['docs']['count']}")
    print(f"    Store size:   {idx_stats['store']['size_in_bytes'] / 1024:.1f} KB")
    print(f"    Segments:     {idx_stats['segments']['count']}")
    print(f"    Segment mem:  {idx_stats['segments']['memory_in_bytes']} bytes")

## 🎯 Interview Tips: When to Use Elasticsearch

### ✅ Good Use Cases
- Full-text search (product search, content search)
- Log analytics and monitoring (ELK stack)
- Geospatial search (Uber, Yelp)
- Faceted search / filtering (e-commerce)
- Autocomplete and suggestions

### ❌ Not a Good Fit
- Primary data store (use PostgreSQL, DynamoDB, etc.)
- Write-heavy workloads (updates are expensive)
- Strong consistency requirements (ES is eventually consistent)
- Simple key-value lookups (use Redis or DynamoDB)

### 🏗️ Architecture Pattern

In interviews, Elasticsearch is typically attached via **Change Data Capture** (CDC)
to an authoritative data store:

```
Client → API → PostgreSQL (source of truth)
                    │
                    └──→ CDC (Debezium/Kafka) → Elasticsearch (search)
```

### 🧠 Key Concepts to Mention
1. **Inverted index** for fast full-text search
2. **Sharding** for horizontal scalability
3. **Replicas** for high availability and read throughput
4. **Eventually consistent** — results may be stale
5. **Denormalize data** for search efficiency
6. **Segment immutability** — great for reads, updates are expensive

## 🧪 Exercises

1. **Shard routing**: Create an index with 5 shards. Index 50 documents and use `_cat/shards` (via the API) to see how documents are distributed
2. **Segment observation**: Create an index, add documents in 5 separate batches with manual refreshes between each. How many segments do you get?
3. **Force merge**: After exercise 2, force merge to 1 segment and compare the storage size before and after
4. **Cluster stats**: Use `es.cluster.stats()` to explore overall cluster resource usage

In [ ]:
# Exercise space — try your experiments here!


## 🎯 Key Takeaways

1. **Shards** split an index across nodes → horizontal scaling + parallel search
2. **Replicas** copy shards to other nodes → fault tolerance + read throughput
3. **Node types**: Master (admin), Data (storage), Coordinating (routing), Ingest (transform)
4. **Lucene segments** are immutable → fast writes, safe caching, easy recovery
5. **Inverted index** maps terms → documents → O(1) lookup instead of O(n) scan
6. **Doc values** store columnar data → fast sorting and aggregations
7. **Refresh interval** controls how quickly indexed data becomes searchable
8. In interviews: ES is for **search**, not as a primary database. Use CDC to keep it in sync.

🎉 **Congratulations!** You've completed the Elasticsearch deep dive.
You now understand how to use Elasticsearch (indexing, searching, analyzers,
aggregations) and how it works under the hood (shards, segments, inverted index).

## 🧹 Cleanup

In [ ]:
# Clean up all demo indices
demo_indices = [
    "demo_1shard", "demo_3shards", "demo_5shards",
    "replica_demo", "multi_shard_demo", "segment_demo",
    "inverted_index_demo", "perf_1s", "perf_5s", "perf_30s", "perf_disabled"
]

for idx in demo_indices:
    if es.indices.exists(index=idx):
        es.indices.delete(index=idx)
        print(f"🗑️  Deleted '{idx}'")

print("\n✅ All demo indices cleaned up")